# Train v5 — standalone notebook

Self-contained training notebook:
- no imports from local `bev_v*.py` files;
- path resolution works with Kaggle-style dataset roots;
- low-coverage train filtering built in;
- test-matched validation split;
- calibration-aware + rover-specialist model.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, copy, time, json, random
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from torchvision import transforms
from PIL import Image, ImageFile
from tqdm import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True

DATA_TRAIN = Path('./autonomy_yandex_dataset_train/')
DATA_VAL   = Path('./autonomy_yandex_dataset_val/')
DATA_TEST  = Path('./autonomy_yandex_dataset_test/')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

## Definitions

In [ ]:
CAMERA_NAMES = [
    '/camera/inner/frontal/middle',
    '/camera/inner/frontal/far',
    '/side/left/forward',
    '/side/right/forward',
]
INTRINSICS_NAMES = [n + '/intrinsic_params' for n in CAMERA_NAMES]
CAR2CAM_NAMES = [n + '/car_to_cam' for n in CAMERA_NAMES]
GT_NAME = 'gt_occupancy_grid'

BEV_H, BEV_W = 188, 126
BEV_RES = 0.8
X_RANGE = (0.0, BEV_H * BEV_RES)
Y_RANGE = (-BEV_W * BEV_RES / 2, BEV_W * BEV_RES / 2)
Z_LEVELS = (0.3, 1.0, 2.0, 3.0)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def resolve_info_path(base_dir: Path, p):
    p = Path(p)
    if p.is_absolute() and p.exists():
        return p
    if p.exists():
        return p
    cand = base_dir / p
    if cand.exists():
        return cand
    cand = base_dir.parent / p
    if cand.exists():
        return cand
    return base_dir.parent / p


def build_rover_vocab(*info_csvs):
    rovers = set()
    for p in info_csvs:
        df = pd.read_csv(p, index_col=0)
        if 'rover' in df.columns:
            rovers.update(df['rover'].unique())
    return {r: i for i, r in enumerate(sorted(rovers))}


def build_specialist_vocab(train_info_csv, test_info_csv, min_train_count=40, topk_test_rovers=12):
    train_info = pd.read_csv(train_info_csv, index_col=0)
    test_info = pd.read_csv(test_info_csv, index_col=0)
    train_counts = Counter(train_info['rover'])
    test_counts = Counter(test_info['rover'])
    selected = []
    for rover, _ in test_counts.most_common():
        if train_counts.get(rover, 0) >= min_train_count:
            selected.append(rover)
        if len(selected) >= topk_test_rovers:
            break
    return {rover: i for i, rover in enumerate(selected)}


def make_test_matched_split(train_info_csv, test_info_csv, group_cols=('rover', 'ride_date'), holdout_frac=0.2, seed=42, cache_path=None):
    if cache_path is not None and Path(cache_path).exists():
        d = np.load(cache_path)
        return d['train_idx'].tolist(), d['val_idx'].tolist()

    train_info = pd.read_csv(train_info_csv, index_col=0).reset_index(drop=True)
    test_info = pd.read_csv(test_info_csv, index_col=0).reset_index(drop=True)
    rng = np.random.RandomState(seed)

    test_rovers = set(test_info['rover'].unique())
    test_counts = Counter(test_info['rover'])
    test_total = sum(test_counts.values())

    rover_to_groups = defaultdict(list)
    grouped = train_info.groupby(list(group_cols)).groups
    for key in grouped.keys():
        rover = key[0]
        if rover in test_rovers:
            rover_to_groups[rover].append(key)

    val_groups = set()
    for rover, groups in rover_to_groups.items():
        groups = list(groups)
        rng.shuffle(groups)
        test_weight = test_counts[rover] / max(test_total, 1)
        rover_frac = min(0.5, max(0.1, holdout_frac * (0.75 + 2.0 * test_weight)))
        n_holdout = max(1, int(round(len(groups) * rover_frac)))
        val_groups.update(groups[:n_holdout])

    train_idx, val_idx = [], []
    for i, row in train_info.iterrows():
        key = tuple(row[c] for c in group_cols)
        if key in val_groups:
            val_idx.append(i)
        else:
            train_idx.append(i)

    if cache_path is not None:
        Path(cache_path).parent.mkdir(parents=True, exist_ok=True)
        np.savez(cache_path, train_idx=np.array(train_idx, dtype=np.int64), val_idx=np.array(val_idx, dtype=np.int64))
    return train_idx, val_idx


def collect_low_coverage_idx(info_csv, coverage_thr=0.01):
    info_csv = Path(info_csv)
    base_dir = info_csv.parent
    info_df = pd.read_csv(info_csv, index_col=0).reset_index(drop=True)
    rows, bad_idx = [], []
    for idx, row in info_df.iterrows():
        gt_path = resolve_info_path(base_dir, row[GT_NAME])
        gt = np.load(gt_path).squeeze()
        valid = (gt != 255)
        coverage = float(valid.mean())
        pos_frac = float((gt[valid] == 1).mean()) if valid.any() else np.nan
        if coverage <= coverage_thr:
            bad_idx.append(idx)
            rows.append({
                'idx': idx,
                'rover': row.get('rover', ''),
                'ride_date': row.get('ride_date', ''),
                'message_ts': row.get('message_ts', ''),
                'coverage': coverage,
                'valid_count': int(valid.sum()),
                'pos_frac': pos_frac,
                'gt_path': str(gt_path),
            })
    bad_df = pd.DataFrame(rows).sort_values(['coverage', 'valid_count', 'rover', 'message_ts'])
    return set(bad_idx), bad_df


def compute_coverage_weights(info_csv, cache_path=None, alpha=0.5, min_weight=0.1):
    if cache_path is not None:
        cache_path = Path(cache_path)
        if cache_path.exists():
            return np.load(cache_path)
    info_csv = Path(info_csv)
    info = pd.read_csv(info_csv, index_col=0)
    base_dir = info_csv.parent
    coverages = np.zeros(len(info), dtype=np.float32)
    for i, (_, row) in enumerate(info.iterrows()):
        if i % 200 == 0:
            print(f'  scan GT coverage: {i}/{len(info)}')
        gt_path = resolve_info_path(base_dir, row[GT_NAME])
        gt = np.load(gt_path).squeeze()
        coverages[i] = (gt != 255).mean()
    weights = coverages ** alpha + min_weight
    if cache_path is not None:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(cache_path, weights)
    return weights


def _camera_pose_features(car2cam):
    cam2car = np.linalg.inv(car2cam)
    trans = cam2car[:3, 3].astype(np.float32)
    forward = cam2car[:3, 2].astype(np.float32)
    return np.concatenate([trans, forward], axis=0)


def build_rig_features(intrinsics, car2cams, img_hw):
    H_t, W_t = img_hw
    cam_feats = []
    poses = []
    for K, M in zip(intrinsics, car2cams):
        pose = _camera_pose_features(M)
        poses.append(pose)
        cam_feat = np.array([
            K[0, 0] / max(W_t, 1),
            K[1, 1] / max(H_t, 1),
            K[0, 2] / max(W_t, 1),
            K[1, 2] / max(H_t, 1),
            pose[0] / 10.0,
            pose[1] / 10.0,
            pose[2] / 10.0,
            pose[3],
            pose[4],
            pose[5],
        ], dtype=np.float32)
        cam_feats.append(cam_feat)

    cam_feats = np.stack(cam_feats, axis=0)
    poses = np.stack(poses, axis=0)
    left_pose, right_pose = poses[2], poses[3]
    mid_pose, far_pose = poses[0], poses[1]
    global_feat = np.array([
        float(np.mean(cam_feats[:, 0])),
        float(np.std(cam_feats[:, 0])),
        float(np.mean(cam_feats[:, 1])),
        float(np.std(cam_feats[:, 1])),
        float(abs(left_pose[1] - right_pose[1]) / 10.0),
        float(abs(mid_pose[0] - far_pose[0]) / 10.0),
        float(abs(mid_pose[2] - far_pose[2]) / 10.0),
        float(np.mean(poses[:, 0]) / 10.0),
        float(np.mean(poses[:, 1]) / 10.0),
        float(np.mean(poses[:, 2]) / 10.0),
        float(np.std(poses[:, 0]) / 10.0),
        float(np.std(poses[:, 1]) / 10.0),
    ], dtype=np.float32)
    return cam_feats, global_feat


class BEVDatasetV5(Dataset):
    def __init__(self, data_dir, mode='train', img_hw=(384, 704), aug=False, rover_vocab=None, specialist_vocab=None, scale_range=(1.0, 1.15)):
        self.data_dir = Path(data_dir)
        self.mode = mode
        self.img_hw = img_hw
        self.info = pd.read_csv(self.data_dir / 'info.csv', index_col=0)
        self.normalize = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
        self.aug = aug and (mode == 'train')
        self.scale_range = scale_range
        self.rover_vocab = rover_vocab or {}
        self.specialist_vocab = specialist_vocab or {}
        self._unknown_rover = len(self.rover_vocab)
        self._unknown_specialist = len(self.specialist_vocab)

    def __len__(self):
        return len(self.info)

    def _resolve_path(self, p):
        return resolve_info_path(self.data_dir, p)

    def _load_one_camera(self, img_path, intr_path, car2cam_path, scale_aug=1.0):
        img_path = self._resolve_path(img_path)
        intr_path = self._resolve_path(intr_path)
        car2cam_path = self._resolve_path(car2cam_path)
        img = Image.open(img_path).convert('RGB')
        src_W, src_H = img.size
        H_t, W_t = self.img_hw

        s = min(W_t / src_W, H_t / src_H)
        new_W, new_H = int(round(src_W * s)), int(round(src_H * s))
        img_resized = img.resize((new_W, new_H), Image.BILINEAR)
        canvas = Image.new('RGB', (W_t, H_t), (0, 0, 0))
        pad_x = (W_t - new_W) // 2
        pad_y = (H_t - new_H) // 2
        canvas.paste(img_resized, (pad_x, pad_y))

        extra_s = 1.0
        extra_dx = 0
        extra_dy = 0
        if scale_aug > 1.0:
            sH, sW = int(round(H_t * scale_aug)), int(round(W_t * scale_aug))
            canvas = canvas.resize((sW, sH), Image.BILINEAR)
            extra_dx = random.randint(0, sW - W_t)
            extra_dy = random.randint(0, sH - H_t)
            canvas = canvas.crop((extra_dx, extra_dy, extra_dx + W_t, extra_dy + H_t))
            extra_s = scale_aug

        arr = np.array(canvas)
        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)
        img_t = torch.from_numpy(arr).permute(2, 0, 1).float() / 255.0
        img_t = self.normalize(img_t)

        intr_full = np.load(intr_path)
        K = intr_full[:, :3].copy().astype(np.float32)
        K[0, 0] *= s; K[0, 2] *= s
        K[1, 1] *= s; K[1, 2] *= s
        K[0, 2] += pad_x; K[1, 2] += pad_y
        K[0, 0] *= extra_s; K[0, 2] *= extra_s
        K[1, 1] *= extra_s; K[1, 2] *= extra_s
        K[0, 2] -= extra_dx; K[1, 2] -= extra_dy

        car2cam = np.load(car2cam_path).astype(np.float32)
        return img_t, K, car2cam

    def _load_sample(self, idx):
        row = self.info.iloc[idx]
        scale_aug = random.uniform(*self.scale_range) if self.aug else 1.0

        imgs, Ks, M = [], [], []
        for cam, intr, c2c in zip(CAMERA_NAMES, INTRINSICS_NAMES, CAR2CAM_NAMES):
            img_t, K, c = self._load_one_camera(row[cam], row[intr], row[c2c], scale_aug=scale_aug)
            imgs.append(img_t)
            Ks.append(torch.from_numpy(K))
            M.append(torch.from_numpy(c))

        images = torch.stack(imgs, dim=0)
        intrinsics = torch.stack(Ks, dim=0)
        car2cams = torch.stack(M, dim=0)

        cam_rig, global_rig = build_rig_features(intrinsics.numpy().astype(np.float32), car2cams.numpy().astype(np.float32), self.img_hw)
        rover = row.get('rover', '?')
        rover_id = self.rover_vocab.get(rover, self._unknown_rover)
        specialist_id = self.specialist_vocab.get(rover, self._unknown_specialist)

        out = {
            'images': images,
            'intrinsics': intrinsics,
            'car2cams': car2cams,
            'rover_id': torch.tensor(rover_id, dtype=torch.long),
            'specialist_id': torch.tensor(specialist_id, dtype=torch.long),
            'cam_rig': torch.from_numpy(cam_rig),
            'global_rig': torch.from_numpy(global_rig),
            'info_idx': idx,
        }
        if self.mode == 'test':
            return out
        gt_path = self._resolve_path(row[GT_NAME])
        gt = np.load(gt_path).squeeze()
        gt = np.where(gt < 0, 255, gt).astype(np.int64)
        out['gt'] = torch.from_numpy(gt).unsqueeze(0)
        return out

    def __getitem__(self, idx):
        max_tries = 5
        last_err = None
        for k in range(max_tries):
            try_idx = (idx + k) % len(self.info)
            try:
                return self._load_sample(try_idx)
            except (OSError, ValueError, FileNotFoundError) as e:
                last_err = e
                continue
        if self.mode == 'test':
            H, W = self.img_hw
            return {
                'images': torch.zeros(4, 3, H, W),
                'intrinsics': torch.eye(3).unsqueeze(0).repeat(4, 1, 1),
                'car2cams': torch.eye(4).unsqueeze(0).repeat(4, 1, 1),
                'rover_id': torch.tensor(0, dtype=torch.long),
                'specialist_id': torch.tensor(0, dtype=torch.long),
                'cam_rig': torch.zeros(4, 10),
                'global_rig': torch.zeros(12),
                'info_idx': idx,
            }
        raise RuntimeError(f'Failed to load any sample after {max_tries} tries starting at {idx}: {last_err}')


class _UNetBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.conv(x)


class SmallUNet(nn.Module):
    def __init__(self, in_c, base_c=32, out_c=1):
        super().__init__()
        self.in_proj = nn.Conv2d(in_c, base_c, 1)
        c = [base_c, base_c * 2, base_c * 4, base_c * 8]
        self.enc1 = _UNetBlock(c[0], c[0])
        self.enc2 = _UNetBlock(c[0], c[1])
        self.enc3 = _UNetBlock(c[1], c[2])
        self.bot = _UNetBlock(c[2], c[3])
        self.dec3 = _UNetBlock(c[3] + c[2], c[2])
        self.dec2 = _UNetBlock(c[2] + c[1], c[1])
        self.dec1 = _UNetBlock(c[1] + c[0], c[0])
        self.out = nn.Conv2d(c[0], out_c, 1)
        self.pool = nn.MaxPool2d(2)
    def _up(self, x, ref):
        return F.interpolate(x, size=ref.shape[-2:], mode='bilinear', align_corners=False)
    def forward(self, x):
        x = self.in_proj(x)
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bot(self.pool(e3))
        d3 = self.dec3(torch.cat([self._up(b, e3), e3], 1))
        d2 = self.dec2(torch.cat([self._up(d3, e2), e2], 1))
        d1 = self.dec1(torch.cat([self._up(d2, e1), e1], 1))
        return self.out(d1)


class _ResNet34Backbone(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = torchvision.models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        rn = torchvision.models.resnet34(weights=weights)
        self.stem = nn.Sequential(rn.conv1, rn.bn1, rn.relu, rn.maxpool)
        self.layer1 = rn.layer1
        self.layer2 = rn.layer2
        self.proj = nn.Conv2d(128, 128, 1)
    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        return self.proj(x)


class _FiLM(nn.Module):
    def __init__(self, cond_dim, feat_dim, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, feat_dim * 2),
        )
    def forward(self, cond):
        gamma_beta = self.net(cond)
        return torch.chunk(gamma_beta, 2, dim=-1)


class MultiCamBEVv5(nn.Module):
    def __init__(self, num_rovers, num_specialists, rover_emb_dim=24, specialist_emb_dim=16, n_cameras=4):
        super().__init__()
        self.n_cameras = n_cameras
        self.bev_h, self.bev_w = BEV_H, BEV_W
        self.x_range, self.y_range = X_RANGE, Y_RANGE
        self.z_levels = Z_LEVELS

        self.backbone = _ResNet34Backbone(pretrained=True)
        self.feat_proj = nn.Conv2d(128, 64, 1)
        self.cam_film = _FiLM(cond_dim=10, feat_dim=64)

        self.num_rovers = num_rovers + 1
        self.num_specialists = num_specialists + 1
        self.rover_embed = nn.Embedding(self.num_rovers, rover_emb_dim)
        self.specialist_embed = nn.Embedding(self.num_specialists, specialist_emb_dim)
        nn.init.normal_(self.rover_embed.weight, std=0.02)
        nn.init.normal_(self.specialist_embed.weight, std=0.02)

        bev_cond_dim = 12 + rover_emb_dim + specialist_emb_dim
        self.bev_film = _FiLM(cond_dim=bev_cond_dim, feat_dim=64 * len(self.z_levels), hidden_dim=96)
        self.specialist_gate = nn.Sequential(
            nn.Linear(bev_cond_dim, 32),
            nn.ReLU(inplace=True),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )

        self.register_buffer('ego_voxels', self._make_ego_voxels(), persistent=False)
        self.bev_decoder = SmallUNet(in_c=64 * len(self.z_levels), base_c=32, out_c=1)

    def _make_ego_voxels(self):
        xs = torch.linspace(self.x_range[0] + BEV_RES / 2, self.x_range[1] - BEV_RES / 2, self.bev_h)
        ys = torch.linspace(self.y_range[0] + BEV_RES / 2, self.y_range[1] - BEV_RES / 2, self.bev_w)
        zs = torch.tensor(self.z_levels, dtype=torch.float32)
        Z, X, Y = torch.meshgrid(zs, xs, ys, indexing='ij')
        ones = torch.ones_like(X)
        return torch.stack([X, Y, Z, ones], dim=-1)

    def forward(self, images, intrinsics, car2cams, rover_ids=None, cam_rig=None, global_rig=None, specialist_ids=None):
        B, N, C_, Hi, Wi = images.shape
        imgs_flat = images.reshape(B * N, C_, Hi, Wi)
        feat = self.backbone(imgs_flat)
        feat = self.feat_proj(feat)
        Hf, Wf = feat.shape[-2:]
        feat = feat.reshape(B, N, 64, Hf, Wf)

        if cam_rig is not None:
            gamma, beta = self.cam_film(cam_rig.reshape(B * N, -1))
            gamma = gamma.view(B, N, 64, 1, 1)
            beta = beta.view(B, N, 64, 1, 1)
            feat = feat * (1.0 + 0.1 * gamma) + 0.1 * beta

        Z, H, W, _ = self.ego_voxels.shape
        V = Z * H * W
        voxels = self.ego_voxels.reshape(-1, 4).unsqueeze(0).unsqueeze(0).expand(B, N, -1, -1)
        p_cam = torch.einsum('bnij,bnvj->bniv', car2cams, voxels)
        p_cam_3d = p_cam[:, :, :3]
        uv_homog = torch.einsum('bnij,bnjv->bniv', intrinsics, p_cam_3d)
        z = uv_homog[:, :, 2]
        u = uv_homog[:, :, 0] / z.clamp(min=1e-3)
        v = uv_homog[:, :, 1] / z.clamp(min=1e-3)
        u_n = 2.0 * u / Wi - 1.0
        v_n = 2.0 * v / Hi - 1.0
        valid = (z > 0.1) & (u_n.abs() <= 1.0) & (v_n.abs() <= 1.0)

        grid = torch.stack([u_n, v_n], dim=-1).reshape(B * N, V, 1, 2)
        feat_flat = feat.reshape(B * N, 64, Hf, Wf)
        sampled = F.grid_sample(feat_flat, grid, mode='bilinear', padding_mode='zeros', align_corners=False)
        sampled = sampled.squeeze(-1).reshape(B, N, 64, V)

        valid_f = valid.float().unsqueeze(2)
        sampled = sampled * valid_f
        agg = sampled.sum(dim=1) / valid_f.sum(dim=1).clamp(min=1.0)
        agg = agg.reshape(B, 64, Z, H, W).reshape(B, 64 * Z, H, W)

        if rover_ids is None:
            rover_ids = torch.zeros(B, dtype=torch.long, device=agg.device)
        if specialist_ids is None:
            specialist_ids = torch.zeros(B, dtype=torch.long, device=agg.device)
        if global_rig is None:
            global_rig = torch.zeros(B, 12, dtype=agg.dtype, device=agg.device)

        rover_emb = self.rover_embed(rover_ids)
        specialist_emb = self.specialist_embed(specialist_ids)
        bev_cond = torch.cat([global_rig, rover_emb, specialist_emb], dim=-1)
        gamma, beta = self.bev_film(bev_cond)
        gamma = gamma.view(B, -1, 1, 1)
        beta = beta.view(B, -1, 1, 1)
        gate = self.specialist_gate(bev_cond).view(B, 1, 1, 1)
        agg = agg * (1.0 + 0.1 * gamma) + 0.1 * gate * beta

        return self.bev_decoder(agg)


def _lovasz_grad(gt_sorted):
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1.0 - gt_sorted).float().cumsum(0)
    jaccard = 1.0 - intersection / union
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard


def lovasz_hinge_flat(logits, labels):
    if labels.numel() == 0:
        return logits.sum() * 0.0
    signs = 2.0 * labels.float() - 1.0
    errors = 1.0 - logits * signs
    errors_sorted, perm = torch.sort(errors, dim=0, descending=True)
    gt_sorted = labels[perm]
    grad = _lovasz_grad(gt_sorted)
    return torch.dot(F.relu(errors_sorted), grad)


class CompoundLossV2(nn.Module):
    def __init__(self, pos_weight=5.0, weight_bce=0.5, weight_dice=0.3, weight_lovasz=0.2, ignore_value=255):
        super().__init__()
        self.register_buffer('pos_weight', torch.tensor([pos_weight]))
        self.weight_bce = weight_bce
        self.weight_dice = weight_dice
        self.weight_lovasz = weight_lovasz
        self.ignore_value = ignore_value
    def forward(self, logits, gt):
        valid = (gt != self.ignore_value)
        gt_f = (gt == 1).float()
        bce_per = F.binary_cross_entropy_with_logits(logits, gt_f, pos_weight=self.pos_weight, reduction='none')
        bce = (bce_per * valid.float()).sum() / valid.float().sum().clamp(min=1.0)
        prob = torch.sigmoid(logits) * valid.float()
        gt_d = gt_f * valid.float()
        inter = (prob * gt_d).sum(dim=(1, 2, 3))
        denom = prob.sum(dim=(1, 2, 3)) + gt_d.sum(dim=(1, 2, 3))
        dice = (1.0 - (2 * inter + 1) / (denom + 1)).mean()
        lov_logits = logits[valid]
        lov_gt = gt_f[valid]
        lov = lovasz_hinge_flat(lov_logits, lov_gt) if lov_logits.numel() > 0 else logits.sum() * 0.0
        total = self.weight_bce * bce + self.weight_dice * dice + self.weight_lovasz * lov
        return total, {'bce': bce.item(), 'dice': dice.item(), 'lovasz': lov.item()}


@torch.no_grad()
def iou_binary_batch(logits, gt, threshold=0.5, ignore_value=255):
    pred = (torch.sigmoid(logits) > threshold).long()
    valid = (gt != ignore_value)
    pred = pred * valid.long()
    gt_b = (gt == 1).long() * valid.long()
    inter = ((pred == 1) & (gt_b == 1)).sum().item()
    union = ((pred == 1) | (gt_b == 1)).sum().item()
    return inter, union


## Config

In [ ]:
cfg = {
    'run_dir': './runs/v5',
    'img_hw': (384, 704),
    'batch_size': 8,
    'grad_accum': 4,
    'epochs': 24,
    'lr_backbone': 3e-5,
    'lr_head': 3e-4,
    'weight_decay': 1e-4,
    'pos_weight': 5.0,
    'num_workers': 4,
    'seed': 42,
    'warmup_epochs': 4,
    'ema_decay': 0.997,
    'holdout_frac': 0.20,
    'topk_specialists': 12,
    'min_train_count': 40,
    'use_sampler': True,
    'drop_low_coverage_train': True,
    'low_coverage_thr': 0.01,
}

RUN_DIR = Path(cfg['run_dir'])
RUN_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(cfg['seed'])
np.random.seed(cfg['seed'])
random.seed(cfg['seed'])

with open(RUN_DIR / 'config.json', 'w') as f:
    json.dump({k: str(v) for k, v in cfg.items()}, f, indent=2)
print(json.dumps({k: str(v) for k, v in cfg.items()}, indent=2))

## Rover vocab + specialists

In [ ]:
rover_vocab = build_rover_vocab(DATA_TRAIN / 'info.csv', DATA_VAL / 'info.csv', DATA_TEST / 'info.csv')
specialist_vocab = build_specialist_vocab(
    DATA_TRAIN / 'info.csv',
    DATA_TEST / 'info.csv',
    min_train_count=cfg['min_train_count'],
    topk_test_rovers=cfg['topk_specialists'],
)
print(f'rover vocab size: {len(rover_vocab)}')
print(f'specialists: {len(specialist_vocab)}')
print(json.dumps(specialist_vocab, indent=2, ensure_ascii=False))

## Split + low-coverage filter

In [ ]:
train_idx, val_idx = make_test_matched_split(
    DATA_TRAIN / 'info.csv',
    DATA_TEST / 'info.csv',
    holdout_frac=cfg['holdout_frac'],
    seed=cfg['seed'],
    cache_path=RUN_DIR / 'test_matched_split.npz',
)

bad_train_idx = set()
bad_train_df = pd.DataFrame()
if cfg['drop_low_coverage_train']:
    bad_train_idx, bad_train_df = collect_low_coverage_idx(DATA_TRAIN / 'info.csv', coverage_thr=cfg['low_coverage_thr'])
    bad_train_df.to_csv(RUN_DIR / 'bad_train_low_coverage.csv', index=False)
    with open(RUN_DIR / 'bad_train_idx.json', 'w') as f:
        json.dump(sorted(list(bad_train_idx)), f, indent=2)
    before = len(train_idx)
    train_idx = [i for i in train_idx if i not in bad_train_idx]
    print(f'low-coverage filter removed {before - len(train_idx)} train samples from matched-train split')
    print(f'global bad train samples: {len(bad_train_idx)}')
    display(bad_train_df.head(20))

print(f'train indices after filter: {len(train_idx)}')
print(f'test-matched val indices: {len(val_idx)}')

## Datasets / loaders

In [ ]:
ds_train_plain = BEVDatasetV5(DATA_TRAIN, mode='train', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab, specialist_vocab=specialist_vocab)
ds_train_aug = BEVDatasetV5(DATA_TRAIN, mode='train', img_hw=cfg['img_hw'], aug=True, rover_vocab=rover_vocab, specialist_vocab=specialist_vocab)
ds_off_val = BEVDatasetV5(DATA_VAL, mode='val', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab, specialist_vocab=specialist_vocab)

ds_train_warm = Subset(ds_train_plain, train_idx)
ds_train = Subset(ds_train_aug, train_idx)
ds_match_val = Subset(ds_train_plain, val_idx)

sampler = None
shuffle = True
if cfg['use_sampler']:
    weights_all = compute_coverage_weights(DATA_TRAIN / 'info.csv', cache_path=RUN_DIR / 'coverage_weights.npy')
    weights = torch.as_tensor(weights_all[train_idx], dtype=torch.double)
    sampler = WeightedRandomSampler(weights=weights, num_samples=len(train_idx), replacement=True)
    shuffle = False

loader_warm = DataLoader(ds_train_warm, batch_size=cfg['batch_size'], shuffle=shuffle if sampler is None else False, sampler=sampler, num_workers=cfg['num_workers'], pin_memory=(device.type=='cuda'), drop_last=True)
loader_train = DataLoader(ds_train, batch_size=cfg['batch_size'], shuffle=shuffle if sampler is None else False, sampler=sampler, num_workers=cfg['num_workers'], pin_memory=(device.type=='cuda'), drop_last=True)
loader_match_val = DataLoader(ds_match_val, batch_size=cfg['batch_size'], shuffle=False, num_workers=cfg['num_workers'])
loader_off_val = DataLoader(ds_off_val, batch_size=cfg['batch_size'], shuffle=False, num_workers=cfg['num_workers'])

sample = ds_train_plain[0]
for k, v in sample.items():
    if isinstance(v, torch.Tensor):
        print(k, tuple(v.shape), v.dtype)
    else:
        print(k, type(v), v)

## Model / optimizer / EMA

In [ ]:
model = MultiCamBEVv5(num_rovers=len(rover_vocab), num_specialists=len(specialist_vocab)).to(device)
n_total = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'total params: {n_total/1e6:.2f}M')
print(f'trainable params: {n_trainable/1e6:.2f}M')

backbone_params = list(model.backbone.parameters())
other_params = [p for n, p in model.named_parameters() if not n.startswith('backbone.')]
optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': cfg['lr_backbone']},
    {'params': other_params, 'lr': cfg['lr_head']},
], weight_decay=cfg['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'], eta_min=1e-6)
loss_fn = CompoundLossV2(pos_weight=cfg['pos_weight'], weight_bce=0.5, weight_dice=0.3, weight_lovasz=0.2).to(device)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

ema_model = copy.deepcopy(model).eval()
for p in ema_model.parameters():
    p.requires_grad = False

@torch.no_grad()
def update_ema(ema, m, decay):
    msd = m.state_dict()
    for k, v in ema.state_dict().items():
        if v.dtype.is_floating_point:
            v.mul_(decay).add_(msd[k], alpha=1 - decay)
        else:
            v.copy_(msd[k])

## Resume

In [ ]:
start_epoch = 0
best_iou = 0.0
best_ema_iou = 0.0
if (RUN_DIR / 'last.pt').exists():
    ck = torch.load(RUN_DIR / 'last.pt', map_location=device)
    model.load_state_dict(ck['model'], strict=False)
    ema_model.load_state_dict(ck['ema'], strict=False)
    optimizer.load_state_dict(ck['opt'])
    scheduler.load_state_dict(ck['sched'])
    start_epoch = ck['epoch'] + 1
    best_iou = ck.get('best_iou', 0.0)
    best_ema_iou = ck.get('best_ema_iou', 0.0)
    print(f'resumed from epoch {start_epoch}, best={best_iou:.4f}, best_ema={best_ema_iou:.4f}')
else:
    print('fresh start')

## Eval helper

In [ ]:
@torch.no_grad()
def evaluate(m, loader, threshold=0.5):
    m.eval()
    inter, union = 0, 0
    for batch in loader:
        imgs = batch['images'].to(device)
        intr = batch['intrinsics'].to(device)
        c2c = batch['car2cams'].to(device)
        rov = batch['rover_id'].to(device)
        cam_rig = batch['cam_rig'].to(device)
        global_rig = batch['global_rig'].to(device)
        specialist = batch['specialist_id'].to(device)
        gt = batch['gt'].to(device)
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            logits = m(imgs, intr, c2c, rov, cam_rig, global_rig, specialist)
        i, u = iou_binary_batch(logits.float(), gt, threshold=threshold)
        inter += i; union += u
    return inter / max(union, 1)

## Train loop

In [ ]:
log = []
t0 = time.time()

for epoch in range(start_epoch, cfg['epochs']):
    model.train()
    train_loader = loader_warm if epoch < cfg['warmup_epochs'] else loader_train
    state = 'WARM' if epoch < cfg['warmup_epochs'] else 'AUG'

    losses = []
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(train_loader, desc=f'ep{epoch:02d} [{state}]')
    for step, batch in enumerate(pbar):
        imgs = batch['images'].to(device)
        intr = batch['intrinsics'].to(device)
        c2c = batch['car2cams'].to(device)
        rov = batch['rover_id'].to(device)
        cam_rig = batch['cam_rig'].to(device)
        global_rig = batch['global_rig'].to(device)
        specialist = batch['specialist_id'].to(device)
        gt = batch['gt'].to(device)

        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            logits = model(imgs, intr, c2c, rov, cam_rig, global_rig, specialist)
            loss, parts = loss_fn(logits, gt)
            loss = loss / cfg['grad_accum']

        scaler.scale(loss).backward()
        if (step + 1) % cfg['grad_accum'] == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            update_ema(ema_model, model, cfg['ema_decay'])

        losses.append(loss.item() * cfg['grad_accum'])
        if step % 20 == 0:
            pbar.set_postfix(loss=f"{np.mean(losses[-50:]):.3f}", bce=f"{parts['bce']:.2f}")

    scheduler.step()

    iou_match = evaluate(model, loader_match_val, threshold=0.5)
    iou_off = evaluate(model, loader_off_val, threshold=0.5)
    iou_match_ema = evaluate(ema_model, loader_match_val, threshold=0.5)
    iou_off_ema = evaluate(ema_model, loader_off_val, threshold=0.5)
    elapsed = (time.time() - t0) / 60

    print(f"ep{epoch:02d} | loss {np.mean(losses):.3f} | match-val: {iou_match:.4f} (ema {iou_match_ema:.4f}) | off-val: {iou_off:.4f} (ema {iou_off_ema:.4f}) | {elapsed:.1f}m")

    row = {
        'epoch': epoch,
        'loss': float(np.mean(losses)),
        'iou_match': iou_match,
        'iou_off': iou_off,
        'iou_match_ema': iou_match_ema,
        'iou_off_ema': iou_off_ema,
        'minutes': elapsed,
    }
    log.append(row)
    pd.DataFrame(log).to_csv(RUN_DIR / 'log.csv', index=False)

    ck = {
        'model_type': 'v5',
        'model': model.state_dict(),
        'ema': ema_model.state_dict(),
        'opt': optimizer.state_dict(),
        'sched': scheduler.state_dict(),
        'epoch': epoch,
        'best_iou': best_iou,
        'best_ema_iou': best_ema_iou,
        'cfg': cfg,
        'val_iou': iou_match,
        'val_iou_off': iou_off,
        'ema_val_iou': iou_match_ema,
        'ema_val_iou_off': iou_off_ema,
        'rover_vocab': rover_vocab,
        'specialist_vocab': specialist_vocab,
    }
    torch.save(ck, RUN_DIR / 'last.pt')
    if iou_match > best_iou:
        best_iou = iou_match
        ck['best_iou'] = best_iou
        torch.save(ck, RUN_DIR / 'best.pt')
        print(f'  new best match-val: {best_iou:.4f}')
    if iou_match_ema > best_ema_iou:
        best_ema_iou = iou_match_ema
        ck['best_ema_iou'] = best_ema_iou
        torch.save(ck, RUN_DIR / 'ema.pt')
        print(f'  new best EMA match-val: {best_ema_iou:.4f}')

print(f'\ndone in {(time.time()-t0)/60:.1f} min')
print(f'best match-val: {best_iou:.4f} | best ema: {best_ema_iou:.4f}')

## After training

Use the standalone [eval_v5.ipynb](/Users/r-shangareev/PyProjects/shine-time/ML2_2026_Competition/eval_v5.ipynb) notebook for:
- threshold sweep;
- honest EMA evaluation;
- submission generation.